***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.3 用于 FFT 的网格化与反网格化](5_3_gridding_and_degridding.ipynb)
    * 下一节：[5.5 小角近似的失效与 $w$ 项](5_5_widefield_effect.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


In [ ]:
HTML('../style/code_toggle.html')


## 5.4 脏图像与可见度权重<a id='imaging:sec:weights'></a>

第 5.2 节给出了脏图像的卷积结构，第 5.3 节说明了不规则可见度如何被网格化到 FFT 平面上。现在还剩下一个会显著改变成像结果的自由度：每个可见度样本到底应该占多大权重。权重不是成像末端的装饰参数，而是测量方程的一部分；它会改变脏波束、图像噪声、角分辨率以及大尺度结构的恢复程度。

在二维 Fourier 近似下，带权采样后的脏图像可写为

$$
I_D(l,m)=\mathcal{F}^{-1}\{W(u,v)V_{\rm obs}(u,v)\},
$$

其中 $W(u,v)$ 是加权后的采样函数。利用卷积定理，得到

$$
I_D(l,m)=B_D(l,m)*I_{\rm app}(l,m),
\qquad
B_D(l,m)=\mathcal{F}^{-1}\{W(u,v)\}.
$$

因此，改变权重函数 $W$，就等价于改变脏波束 $B_D$。权重的核心取舍可以概括为：强调密集采样区域通常能提高灵敏度、压低噪声，却会牺牲分辨率；强调稀疏的长基线区域通常能提高分辨率，却会抬高噪声和旁瓣。


In [ ]:
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)


def normalize(arr):
    arr = np.asarray(arr, dtype=float)
    maxval = np.max(np.abs(arr))
    return arr if maxval == 0 else arr / maxval


def make_synthetic_sky(grid_size=160, field_size_deg=1.6):
    axis = (np.arange(grid_size) - grid_size / 2) * (field_size_deg / grid_size)
    ll, mm = np.meshgrid(axis, axis)
    sky = (
        0.90 * np.exp(-((ll + 0.22) ** 2 + (mm - 0.08) ** 2) / (2 * 0.045 ** 2))
        + 0.55 * np.exp(-((ll - 0.10) ** 2 + (mm + 0.18) ** 2) / (2 * 0.080 ** 2))
        + 0.20 * np.exp(-((ll + 0.02) ** 2 + (mm + 0.02) ** 2) / (2 * 0.220 ** 2))
    )
    for l0, m0, flux in [(0.30, -0.27, 0.32), (-0.34, 0.25, 0.24), (0.08, 0.03, 0.18)]:
        ix = np.argmin(np.abs(axis - l0))
        iy = np.argmin(np.abs(axis - m0))
        sky[iy, ix] += flux
    return axis, normalize(sky)


def make_uv_samples(grid_size=160, n_pairs=4800, seed=11, dense_core=True):
    rng = np.random.default_rng(seed)
    max_radius = 0.45 * grid_size
    angles = rng.uniform(0.0, 2.0 * np.pi, n_pairs)
    if dense_core:
        n_core = int(0.70 * n_pairs)
        core = max_radius * 0.25 * np.sqrt(rng.uniform(0.0, 1.0, n_core))
        tail = max_radius * np.sqrt(rng.uniform(0.12, 1.0, n_pairs - n_core))
        radii = np.concatenate([core, tail])
        rng.shuffle(radii)
    else:
        radii = max_radius * np.sqrt(rng.uniform(0.04, 1.0, n_pairs))
    u = np.rint(radii * np.cos(angles)).astype(int)
    v = np.rint(radii * np.sin(angles)).astype(int)
    keep = (np.abs(u) < grid_size // 2 - 2) & (np.abs(v) < grid_size // 2 - 2) & ((u != 0) | (v != 0))
    half = np.column_stack([u[keep], v[keep]])
    return np.vstack([half, -half])


def make_observed_vis(sky, coords, noise_scale=0.012, seed=12):
    rng = np.random.default_rng(seed)
    n = sky.shape[0]
    center = n // 2
    vis_grid = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(sky)))
    samples = vis_grid[coords[:, 1] + center, coords[:, 0] + center]
    noise = noise_scale * np.max(np.abs(samples)) * (rng.normal(size=samples.size) + 1j * rng.normal(size=samples.size))
    return samples + noise


def density_grid(coords, grid_size):
    center = grid_size // 2
    density = np.zeros((grid_size, grid_size), dtype=float)
    np.add.at(density, (coords[:, 1] + center, coords[:, 0] + center), 1.0)
    return density


def density_per_sample(coords, density):
    center = density.shape[0] // 2
    return density[coords[:, 1] + center, coords[:, 0] + center]


def sample_weights(coords, density, mode='natural', robust=0.0):
    d = density_per_sample(coords, density)
    if mode == 'natural':
        weights = np.ones_like(d)
    elif mode == 'uniform':
        weights = 1.0 / np.maximum(d, 1.0)
    elif mode == 'robust':
        sampled_density = d[d > 0]
        scale = np.median(sampled_density) * 10.0 ** robust
        weights = 1.0 / (1.0 + d / max(scale, 1e-12))
    else:
        raise ValueError(mode)
    return weights / np.mean(weights)


def gaussian_taper(coords, sigma_uv):
    r2 = coords[:, 0] ** 2 + coords[:, 1] ** 2
    taper = np.exp(-0.5 * r2 / sigma_uv ** 2)
    return taper / np.mean(taper)


def reliability_weights(coords, grid_size, bad_quadrant_factor=0.35):
    weights = np.ones(coords.shape[0], dtype=float)
    bad = (coords[:, 0] > 0) & (coords[:, 1] > 0)
    weights[bad] *= bad_quadrant_factor
    return weights / np.mean(weights)


def image_products(sky, coords, vis_samples, weights):
    n = sky.shape[0]
    center = n // 2
    weighted_vis = np.zeros((n, n), dtype=complex)
    weighted_sampling = np.zeros((n, n), dtype=complex)
    np.add.at(weighted_vis, (coords[:, 1] + center, coords[:, 0] + center), vis_samples * weights)
    np.add.at(weighted_sampling, (coords[:, 1] + center, coords[:, 0] + center), weights)
    dirty = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(weighted_vis))).real
    psf = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(weighted_sampling))).real
    peak = np.max(psf)
    if peak != 0:
        dirty = dirty / peak
        psf = psf / peak
    return dirty, psf, weighted_sampling.real


def central_crop(arr, half_width=42):
    c = arr.shape[0] // 2
    return arr[c - half_width:c + half_width, c - half_width:c + half_width]


def half_power_width(psf):
    c = psf.shape[0] // 2
    profile = psf[c]
    above = np.where(profile > 0.5)[0]
    return int(above[-1] - above[0] + 1) if above.size else 0


def background_rms(image, sky):
    mask = sky < 0.04
    return float(np.std(image[mask]))


In [ ]:
axis, sky = make_synthetic_sky()
coords = make_uv_samples(grid_size=sky.shape[0], dense_core=True)
density = density_grid(coords, sky.shape[0])
vis_samples = make_observed_vis(sky, coords)

natural_w = sample_weights(coords, density, mode='natural')
uniform_w = sample_weights(coords, density, mode='uniform')
robust0_w = sample_weights(coords, density, mode='robust', robust=0.0)

weight_sets = {
    'natural': natural_w,
    'robust 0': robust0_w,
    'uniform': uniform_w,
}
products = {}
for name, weights in weight_sets.items():
    dirty, psf, weighted_sampling = image_products(sky, coords, vis_samples, weights)
    products[name] = {'dirty': dirty, 'psf': psf, 'sampling': weighted_sampling, 'weights': weights}


### 5.4.1 脏图像中的权重

把所有影响因子合并起来，常把加权采样函数写成

$$
W(u,v)=R(u,v)\,T(u,v)\,D(u,v)\,S(u,v).
$$

这里 $S$ 是几何采样函数，$D$ 是密度权重，$T$ 是锥化或 taper，$R$ 是测量可靠性权重。这个分解并不唯一，但它很有用：$D$ 主要处理 `uv` 平面采样密度不均的问题，$T$ 按科学目标选择角尺度，$R$ 则反映不同测量的噪声可靠性。它们全部乘在可见度域中，因此都会通过 Fourier 变换改变图像域中的脏波束。


In [ ]:
extent = [axis[0], axis[-1], axis[0], axis[-1]]
uv_extent = [-sky.shape[0] / 2, sky.shape[0] / 2, -sky.shape[0] / 2, sky.shape[0] / 2]

fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.2))
axes[0, 0].imshow(sky, cmap='magma', origin='lower', extent=extent)
axes[0, 0].set_title('model sky')
axes[0, 0].set_xlabel('l (deg)')
axes[0, 0].set_ylabel('m (deg)')

axes[1, 0].imshow(np.log10(density + 1.0), cmap='viridis', origin='lower', extent=uv_extent)
axes[1, 0].set_title('sample density')
axes[1, 0].set_xlabel('u (grid units)')
axes[1, 0].set_ylabel('v (grid units)')

for col, name in enumerate(['natural', 'robust 0', 'uniform'], start=0):
    if col == 0:
        target_ax = axes[0, 1]
        psf_ax = axes[1, 1]
    elif col == 1:
        target_ax = axes[0, 2]
        psf_ax = axes[1, 2]
    else:
        continue
    target_ax.imshow(products[name]['dirty'], cmap='magma', origin='lower', extent=extent)
    target_ax.set_title(f'{name} dirty image')
    target_ax.set_xlabel('l (deg)')
    target_ax.set_ylabel('m (deg)')
    psf_ax.imshow(central_crop(products[name]['psf'], 38), cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    psf_ax.set_title(f'{name} dirty beam')
    psf_ax.set_xticks([])
    psf_ax.set_yticks([])

fig.tight_layout()
fig.savefig(FIG_DIR / 'dirty_image_weighting_overview.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![脏图像、采样密度与权重](figures/dirty_image_weighting_overview.png)

**图 5.4.1** 同一组模拟可见度在不同权重下形成的脏图像和脏波束。`uv` 中心附近采样更密，因此自然权重会更强调短基线；均匀权重则削弱重复采样区域的总权重，使长基线相对更重要。


### 5.4.2 密度权重：自然、均匀与鲁棒权重

若每一次测量都有相同噪声方差，最直接的做法是让每个可见度样本权重相同。这就是自然权重。若某些 `uv` 单元被重复采样很多次，它们的总权重自然更大，因此自然权重会给采样密集区更高的统计影响。它的优点是图像热噪声最低，缺点是常常使短基线和低空间频率占主导，导致脏波束主瓣变宽。

均匀权重则先统计每个 `uv` 单元中的样本数 $N_s(u,v)$，再让该单元内每个样本的权重近似与 $1/N_s$ 成正比。这样，每个被采样单元的总权重更接近一致，稀疏长基线获得更高相对权重。它通常提高角分辨率，但会抬高图像噪声，并可能增强旁瓣。

鲁棒权重或 Briggs 权重位于二者之间。实际软件实现有所不同，但共同目标是用一个连续参数在自然权重和均匀权重之间移动。这里使用一个教学形式：

$$
w_i(R)=\frac{1}{1+N_s(u_i,v_i)/f(R)},
\qquad
f(R)=N_{\rm med}\,10^R.
$$

当 $R$ 较大时，$f(R)$ 大，权重接近自然权重；当 $R$ 较小时，权重近似与 $1/N_s$ 成正比，因而接近均匀权重。这个简化公式不等同于某个具体成像软件的精确定义，但足以展示鲁棒参数所控制的物理取舍。


In [ ]:
robust_values = [-2, -1, 0, 1, 2]
robust_products = {}
for robust in robust_values:
    weights = sample_weights(coords, density, mode='robust', robust=robust)
    dirty, psf, weighted_sampling = image_products(sky, coords, vis_samples, weights)
    robust_products[robust] = {'dirty': dirty, 'psf': psf, 'weights': weights}

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
center = sky.shape[0] // 2
window = slice(center - 30, center + 31)
for name, color in [('natural', '#d62828'), ('uniform', '#1d3557')]:
    axes[0].plot(products[name]['psf'][center, window], lw=2.5, label=name, color=color)
for robust in robust_values:
    axes[0].plot(robust_products[robust]['psf'][center, window], lw=1.6, label=f'R={robust}')
axes[0].set_title('dirty-beam cross sections')
axes[0].set_xlabel('pixel across beam center')
axes[0].set_ylabel('normalized response')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8, frameon=False, ncol=2)

labels = ['natural', 'robust 0', 'uniform']
widths = [half_power_width(products[label]['psf']) for label in labels]
rms_values = [background_rms(products[label]['dirty'], sky) for label in labels]
x = np.arange(len(labels))
ax2 = axes[1]
ax2.bar(x - 0.18, widths, width=0.36, label='half-power width [pix]', color='#457b9d')
ax2b = ax2.twinx()
ax2b.bar(x + 0.18, rms_values, width=0.36, label='background RMS', color='#e76f51')
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_title('resolution-noise tradeoff')
ax2.set_ylabel('beam width [pixels]')
ax2b.set_ylabel('background RMS')
ax2.grid(alpha=0.2, axis='y')

handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(handles1 + handles2, labels1 + labels2, frameon=False, loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'density_weighting_tradeoff.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![密度权重的取舍](figures/density_weighting_tradeoff.png)

**图 5.4.2** 自然权重、均匀权重和鲁棒权重的连续变化。自然权重主瓣更宽但背景 RMS 较低；均匀权重主瓣更窄但噪声更高；鲁棒权重允许在这两个端点之间移动。


### 5.4.3 锥化：按角尺度选择信号

密度权重主要是在采样密度不均时调节灵敏度和分辨率。锥化权重则更像一个有明确科学目标的空间滤波器。若希望提高扩展源的信噪比，可以在 `uv` 平面乘上一个随半径下降的平滑函数，削弱长基线，使最终图像分辨率降低但表面亮度灵敏度提高。最常用的锥化函数之一是高斯：

$$
T(u,v)=\exp\left[-\frac{u^2+v^2}{2\sigma_{uv}^2}\right].
$$

由于高斯的 Fourier 变换仍是高斯，`uv` 域中的高斯锥化对应图像域中的平滑卷积。$$\sigma_{uv}$ 越小，长基线被压得越强，图像域中的平滑尺度越大。这个过程会损失角分辨率，但能更好地显示低表面亮度、较大角尺度的结构。


In [ ]:
sigma_values = [72, 42, 24]
taper_products = {}
base_weights = sample_weights(coords, density, mode='uniform')
for sigma_uv in sigma_values:
    weights = base_weights * gaussian_taper(coords, sigma_uv)
    dirty, psf, weighted_sampling = image_products(sky, coords, vis_samples, weights)
    taper_products[sigma_uv] = {'dirty': dirty, 'psf': psf, 'weights': weights}

fig, axes = plt.subplots(2, 3, figsize=(13.0, 8.0))
for col, sigma_uv in enumerate(sigma_values):
    axes[0, col].imshow(central_crop(taper_products[sigma_uv]['psf'], 40), cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    axes[0, col].set_title(fr'PSF, $\sigma_{{uv}}={sigma_uv}$')
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])
    axes[1, col].imshow(taper_products[sigma_uv]['dirty'], cmap='magma', origin='lower', extent=extent)
    axes[1, col].set_title('tapered dirty image')
    axes[1, col].set_xlabel('l (deg)')
    axes[1, col].set_ylabel('m (deg)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'uv_tapering_effect.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![高斯锥化的影响](figures/uv_tapering_effect.png)

**图 5.4.3** 在均匀权重基础上施加不同宽度的高斯 `uv` 锥化。锥化越强，长基线被压得越多，脏波束主瓣越宽，图像中的细节越少，但扩展结构的相对可见性更高。


### 5.4.4 测量可靠性权重

并非所有可见度样本都同样可靠。系统温度较高、天线有效面积较小、校准质量较差、RFI 标记较多，都会增大某些基线或时间段的噪声。若第 $i$ 个可见度的噪声方差为 $\sigma_i^2$，最常用的统计权重是

$$
R_i\propto \frac{1}{\sigma_i^2}.
$$

这类权重的作用与密度权重不同。密度权重处理的是采样几何造成的重复程度；可靠性权重处理的是测量本身的可信程度。VLBI 和异质阵列尤其需要这一步，因为不同望远镜的灵敏度可能相差很大。若忽略可靠性差异，少数噪声较大的基线可能把结构性条纹带入脏图像；若权重设置过于激进，又可能损失某些方向的 `uv` 覆盖。


In [ ]:
reliable = reliability_weights(coords, sky.shape[0], bad_quadrant_factor=0.25)
rel_dirty, rel_psf, rel_sampling = image_products(sky, coords, vis_samples, natural_w * reliable)
base_dirty, base_psf, base_sampling = image_products(sky, coords, vis_samples, natural_w)

fig, axes = plt.subplots(2, 3, figsize=(13.0, 8.0))
axes[0, 0].imshow(np.log10(base_sampling + 1.0), cmap='viridis', origin='lower', extent=uv_extent)
axes[0, 0].set_title('natural sampling')
axes[0, 1].imshow(np.log10(rel_sampling + 1.0), cmap='viridis', origin='lower', extent=uv_extent)
axes[0, 1].set_title('with reliability downweighting')
axes[0, 2].imshow(np.log10(np.abs(base_sampling - rel_sampling) + 1.0), cmap='magma', origin='lower', extent=uv_extent)
axes[0, 2].set_title('changed weight pattern')

axes[1, 0].imshow(central_crop(base_psf, 40), cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
axes[1, 0].set_title('natural PSF')
axes[1, 1].imshow(central_crop(rel_psf, 40), cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
axes[1, 1].set_title('reliability-weighted PSF')
axes[1, 2].imshow(rel_dirty - base_dirty, cmap='coolwarm', origin='lower', extent=extent)
axes[1, 2].set_title('dirty-image difference')
for ax in axes.flat:
    ax.set_xlabel('')
    ax.set_ylabel('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'reliability_weighting_effect.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![可靠性权重的影响](figures/reliability_weighting_effect.png)

**图 5.4.4** 一个示意性的可靠性权重实验。右上象限的可见度被下调后，采样函数不再保持原来的方向均衡，脏波束和脏图像也随之改变。这说明可靠性权重虽然来自噪声统计，却同样会通过 $W(u,v)$ 改变成像响应。


### 5.4.5 脏图像的局限性

脏图像不是“错误图像”，而是带有已知测量响应的图像。它包含真实天空与脏波束的卷积，也包含噪声、权重、网格化误差和未采样空间频率共同留下的结构。权重可以改善某些方面，却不能凭空补回没有测到的信息：自然权重提高灵敏度但降低分辨率，均匀权重提高分辨率但增加噪声，鲁棒权重提供折中，锥化权重按科学目标选择角尺度，可靠性权重则把测量噪声纳入成像。

去卷积的任务，正是从这个带权脏图像出发，利用脏波束和天空先验来构造更接近真实亮度分布的模型。进入去卷积之前，本章还要处理一个几何近似问题：前面所有公式都基于二维 Fourier 关系，而宽视场和非共面基线会重新引入 $w$ 项。第 5.5 节将讨论这一近似如何失效，以及现代成像如何修正它。


***

* 下一节：[5.5 小角近似的失效与 $w$ 项](5_5_widefield_effect.ipynb)
